### Training with **Colab**

<div style="padding: 20px; margin: 20px 0; background-color: #ff0000; display: flex; flex-direction: column; gap: 10px; align-items: center;">
    <div style="background-color: #000000; display: flex; flex-direction: row; gap: 10px; align-items: center; padding: 10px; border-radius: 5px; justify-content: center;">
        <a href="https://colab.research.google.com/github/sadbinsiddique/Dl-net/blob/main/notebook/3.%20Train.ipynb" target="_blank">
        <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
        </a>
        <a href="https://github.com/settings/tokens" target="_blank" style="display: flex; align-items: center; gap: 5px; text-decoration: none; color: #ffffff;">
        <img src="https://github.githubassets.com/favicons/favicon.svg" alt="GitHub" style="height: 20px; width: 20px;" />
        <span style="font-size: 14px;">Access Token</span>
        </a>
    </div>
</div>

In [ ]:
from getpass import getpass
import os

# Check if running in Colab
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")

if IN_COLAB:
    token = getpass('Enter your GitHub Personal Access Token: ')
    # Go https://github.com/settings/tokens
    !git clone https://{token}@github.com/sadbinsiddique/Dl-net.git
else:
    print("Local environment detected. Skipping git clone.")

In [ ]:
if IN_COLAB:
    os.chdir('/content/Dl-net')
    DATA_PATH = '/content/Dl-net/notebook/'
    print(f"Colab environment setup. Current working directory: {os.getcwd()}")
else:
    DATA_PATH = './'
    print(f"Local environment. Data path: {DATA_PATH}")

print(f"Data will be loaded from: {DATA_PATH}")

### Main Cord Start

In [16]:
import os
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd


In [ ]:
train_df = pd.read_csv(f"{DATA_PATH}train.csv")
test_df = pd.read_csv(f"{DATA_PATH}test.csv")

test = test_df[["filepath","class"]]
train = train_df[["filepath","class"]]

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

In [18]:
test['class'].unique()

<ArrowStringArray>
['neutral', 'angry', 'sad', 'happy', 'fear']
Length: 5, dtype: str

In [19]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [20]:
from torch.utils.data import Dataset
from PIL import Image

class EmotionDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        image = Image.open(
            self.df.loc[idx,"filepath"]
        ).convert("RGB")

        label = int(self.df.loc[idx,"label"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [21]:
from torch.utils.data import DataLoader

train_dataset = EmotionDataset(train, train_transform)
val_dataset = EmotionDataset(test, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

## Alex Net


In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)

num_classes = 5

model.classifier[6] = nn.Linear(
    4096,
    num_classes
)

model = model.to(device)

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [ ]:
from tqdm import tqdm

epochs = 20

for epoch in range(epochs):

    model.train()

    train_loss = 0
    correct = 0
    total = 0

    loop = tqdm(train_loader)

    for images, labels in loop:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

        loop.set_description(f"Epoch {epoch+1}")

        loop.set_postfix(
            loss=loss.item(),
            acc=100*correct/total
        )

    print(
        f"Epoch {epoch+1} | "
        f"Loss={train_loss/len(train_loader):.4f} | "
        f"Accuracy={100*correct/total:.2f}%"
    )

  0%|          | 0/590 [00:00<?, ?it/s]